# LoRA GPT-2 Medium E2E Training on Google Colab

Use this notebook to run the GPT-2 Medium + LoRA + E2E NLG replication on a Colab GPU.

Recommended runtime:

- Runtime > Change runtime type > Hardware accelerator: GPU
- A T4 GPU should be enough for the baseline, though full training can still take several hours.

The notebook clones the project from GitHub, downloads the E2E files, preprocesses them, runs a dry run, then starts training.

## 1. Check GPU

Run this first to confirm Colab assigned a CUDA GPU.

In [1]:
!nvidia-smi

import torch
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

Sat Apr 25 18:40:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Clone The Repo

If you rerun the notebook in the same runtime, this cell updates the existing clone instead of failing.

In [3]:
from getpass import getpass

token = getpass("GitHub token: ")

GitHub token: ··········


In [5]:
%cd /content
!git clone https://$token@github.com/justinlxiang/CS4782-final-project.git
%cd /content/CS4782-final-project/lora-gpt2-medium-e2e
!pwd

/content
fatal: destination path 'CS4782-final-project' already exists and is not an empty directory.
/content/CS4782-final-project/lora-gpt2-medium-e2e
/content/CS4782-final-project/lora-gpt2-medium-e2e


## 3. Install Dependencies

Colab already has PyTorch installed. Installing the requirements should add the remaining packages.

In [6]:
!pip install -q -r requirements.txt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.6 MB/s eta 0:00:00


## 4. Optional: Mount Google Drive

Recommended for long runs. Colab runtimes can disconnect, so copying checkpoints to Drive protects your outputs.

In [7]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/e2e_lora_r4_alpha32"

Mounted at /content/drive


## 5. Download E2E Dataset

These files come from the official Microsoft LoRA repo and are already formatted as `context||completion`.

In [8]:
!mkdir -p data/raw/e2e
!curl -L -o data/raw/e2e/train.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/train.txt
!curl -L -o data/raw/e2e/valid.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/valid.txt
!curl -L -o data/raw/e2e/test.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/test.txt

!wc -l data/raw/e2e/*.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 9398k  100 9398k    0     0  11.0M      0 --:--:-- --:--:-- --:--:-- 11.0M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1170k  100 1170k    0     0  3976k      0 --:--:-- --:--:-- --:--:-- 3966k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1319k  100 1319k    0     0  4293k      0 --:--:-- --:--:-- --:--:-- 4297k
    4693 data/raw/e2e/test.txt
   42061 data/raw/e2e/train.txt
    4672 data/raw/e2e/valid.txt
   51426 total


## 6. Preprocess E2E

This creates tokenized JSONL files under `data/processed/e2e_gpt2/`.

The current preprocessing uses the official-style sequence:

`raw_context + 50256 + leading_space_completion + 50256`

In [10]:
!python scripts/prepare_e2e.py --config configs/e2e_gpt2_medium_lora.yaml
!python - <<'PY2'
import json
from pathlib import Path
path = Path('data/processed/e2e_gpt2/train.jsonl')
example = json.loads(path.read_text().splitlines()[0])
print('prompt:', example['prompt'])
print('first input ids:', example['input_ids'][:20])
print('first labels:', example['labels'][:20])
print('prompt length:', example['prompt_length'])

wrote 42061 examples to /content/CS4782-final-project/lora-gpt2-medium-e2e/data/processed/e2e_gpt2/train.jsonl
wrote 4672 examples to /content/CS4782-final-project/lora-gpt2-medium-e2e/data/processed/e2e_gpt2/valid.jsonl
wrote 4693 examples to /content/CS4782-final-project/lora-gpt2-medium-e2e/data/processed/e2e_gpt2/test.jsonl
/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY2')
prompt: name : The Vaults | Type : pub | price : more than £ 30 | customer rating : 5 out of 5 | near : Café Adriatic
first input ids: [3672, 1058, 383, 21314, 930, 5994, 1058, 2240, 930, 2756, 1058, 517, 621, 4248, 1542, 930, 6491, 7955, 1058, 642]
first labels: [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]
prompt length: 31


## 7. Run Tests And Dry Run

This confirms the code works, LoRA injects into all 24 GPT-2 Medium layers, CUDA works, and a tiny forward pass succeeds.

In [11]:
!python -m pytest
!python scripts/count_params.py --config configs/e2e_gpt2_medium_lora.yaml
!python scripts/train.py --config configs/e2e_gpt2_medium_lora.yaml --dry-run --device cuda --dry-run-forward-pass

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content/CS4782-final-project/lora-gpt2-medium-e2e
configfile: pyproject.toml
testpaths: tests
plugins: langsmith-0.7.30, typeguard-4.5.1, anyio-4.13.0
collected 19 items                                                             

tests/test_checkpointing.py ...                                          [ 15%]
tests/test_data.py .....                                                 [ 42%]
tests/test_evaluate.py .                                                 [ 47%]
tests/test_generation.py ...                                             [ 63%]
tests/test_inject.py .                                                   [ 68%]
tests/test_lora_layers.py ...                                            [ 84%]
tests/test_train.py .                                                    [ 89%]
tests/test_utils.py ..                                  

## 8. Optional: Short Smoke Training

This runs a few real optimizer steps to confirm backward pass and optimizer updates work on CUDA. It is not the full experiment.

In [12]:
!python scripts/train.py \
  --config configs/e2e_gpt2_medium_lora.yaml \
  --smoke-train \
  --device cuda \
  --dry-run-max-examples 80 \
  --dry-run-batch-size 8 \
  --smoke-max-steps 10

Loading weights: 100% 292/292 [00:00<00:00, 895.50it/s, Materializing param=transformer.wte.weight] 
GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
dry_run=False
smoke_train=True
train=False
device=cuda
replaced_modules=24
trainable_parameters=393216
optimizer_param_groups=1
scheduler=LambdaLR
dry_run_dataset=/content/CS4782-final-project/lora-gpt2-medium-e2e/data/processed/e2e_gpt2/train.jsonl
dry_run_examples=80
dry_run_batch_input_shape=(8, 78)
dry_run_batch_label_shape=(8, 78)
smoke_step=1 loss=6.0659
smoke_step=2 loss=5.9105
smoke_step=3 loss=5.6254
smoke_step=4 loss=5.8409
smoke_step=5 loss=5.7177
smoke_step=6 loss=6.0073
smoke_step=7 loss=5.9436
smoke_step=8 loss=5.8291
smoke_step=9 loss=6.0344
smoke_step=10 loss=5.5818
smoke_train_steps=10
sm

## 9. Full Training

This is the paper-style baseline run: GPT-2 Medium, E2E, LoRA rank 4, alpha 32, dropout 0.1, batch size 8, 5 epochs.

Outputs go to:

`outputs/runs/e2e_lora_r4_alpha32/`

Checkpoints are saved every 1000 steps and at the end.

In [ ]:
!python scripts/train.py \
  --config configs/e2e_gpt2_medium_lora.yaml \
  --train \
  --device cuda

Loading weights: 100% 292/292 [00:00<00:00, 1130.44it/s, Materializing param=transformer.wte.weight]
GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
dry_run=False
smoke_train=False
train=True
device=cuda
replaced_modules=24
trainable_parameters=393216
optimizer_param_groups=1
scheduler=LambdaLR
dry_run_dataset=/content/CS4782-final-project/lora-gpt2-medium-e2e/data/processed/e2e_gpt2/train.jsonl
dry_run_examples=2
dry_run_batch_input_shape=(1, 52)
dry_run_batch_label_shape=(1, 52)
train_dataset=/content/CS4782-final-project/lora-gpt2-medium-e2e/data/processed/e2e_gpt2/train.jsonl
train_examples=42061
train_batch_size=8
configured_training_steps=26290
effective_training_steps=26290
output_dir=/content/CS4782-final-project/lora-gpt2-medium-e2e/outputs/r

## 10. Back Up Outputs To Drive

Run this after training to back up training outputs.

In [ ]:
!mkdir -p /content/drive/MyDrive
!cp -r outputs/runs/e2e_lora_r4_alpha32 "$DRIVE_OUTPUT_DIR"
!du -sh "$DRIVE_OUTPUT_DIR"

## 11. Resume From A Checkpoint

If Colab disconnects, point to the latest checkpoint and resume. A resumed run appends to the same `metrics.jsonl` and skips already-completed batches from the checkpointed epoch.

In [ ]:
# Example: change this to the latest checkpoint you have.
RESUME_CHECKPOINT = "outputs/runs/e2e_lora_r4_alpha32/checkpoints/adapter_step_1000.pt"

!python scripts/train.py \
  --config configs/e2e_gpt2_medium_lora.yaml \
  --train \
  --device cuda \
  --resume-checkpoint "$RESUME_CHECKPOINT"

## 12. Inspect Training Logs

Use this to check the final few loss records and available checkpoints.

In [ ]:
!tail -n 10 outputs/runs/e2e_lora_r4_alpha32/metrics.jsonl
!ls -lh outputs/runs/e2e_lora_r4_alpha32/checkpoints | tail